In [26]:
import torch                           # PyTorch 核心库，用于张量计算和模型训练
from torchvision import datasets       # 常用图像数据集，如 MNIST、CIFAR-10
from torchvision import transforms     # 图像预处理和数据增强工具
import torch.nn as nn                  # 神经网络模块，用于定义网络层和损失函数
import torch.optim as optim            # 优化器模块，用于更新模型参数

### 1. 准备数据

In [27]:
train_data = datasets.MNIST(
    root="C:/Users/tangy/PyCharmMiscProject/Datasets/MNIST",  # 数据集保存路径
    train=True,                       # 加载训练集
    transform=transforms.ToTensor(),  # 将图片转换为 Tensor
    download=True                     # 本地没有数据时自动下载
)
test_data = datasets.MNIST(
    root="C:/Users/tangy/PyCharmMiscProject/Datasets/MNIST",
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

In [28]:
batch_size = 100  # 每个批次读取 100 个样本
train_loader = torch.utils.data.DataLoader(
    dataset=train_data,      # 训练数据集
    batch_size=batch_size,   # 每批样本数量
    shuffle=True             # 每轮训练前打乱数据顺序
)
test_loader = torch.utils.data.DataLoader(
    dataset=test_data,       # 测试数据集
    batch_size=batch_size,   # 每批样本数量
    shuffle=False            # 测试时不打乱数据
)

### 2. 定义神经网络

In [29]:
class MLP(nn.Module):  # 定义一个多层感知机（MLP）模型，定义一个名为 MLP 的类，并让它继承 PyTorch 的 nn.Module 类。
    # 定义 MLP 类的初始化方法，input_size输入数据的维度，hidden_size 隐藏层的大小，num_classes 输出分类的数量
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()  # 初始化父类 nn.Module
        self.fc1 = nn.Linear(input_size, hidden_size)       # 第1个全连接层：输入层 → 隐藏层
        self.relu = nn.ReLU()                               # ReLU 激活函数
        self.fc2 = nn.Linear(hidden_size, hidden_size)      # 第2个全连接层：隐藏层 → 隐藏层
        self.fc3 = nn.Linear(hidden_size, num_classes)      # 输出层：隐藏层 → 分类结果

    def forward(self, x):       # 定义数据在网络中的前向传播过程
        out = self.fc1(x)       # 输入数据经过第1个全连接层
        out = self.relu(out)    # 使用 ReLU 激活函数
        out = self.fc2(out)     # 经过第2个全连接层
        out = self.relu(out)    # 再次使用 ReLU 激活函数
        out = self.fc3(out)     # 经过输出层，得到各类别的输出值
        return out              # 返回模型的最终输出

input_size = 28 * 28   # 输入维度：MNIST 图片为 28×28，展开后共 784 个特征
hidden_size = 512      # 隐藏层神经元数量
num_classes = 10       # 输出类别数量：数字 0~9，共 10 类

model = MLP(input_size, hidden_size, num_classes)  # 创建 MLP 模型实例

### 3. 定义损失函数 Loss Function

In [30]:
criterion = nn.CrossEntropyLoss()  # 定义交叉熵损失函数，用于多分类任务中衡量预测结果与真实标签之间的差异

### 4. 定义优化器 Optimizer

In [31]:
learning_rate = 0.001  # 设置学习率，控制每次参数更新的步长
optimizer = optim.Adam(
    model.parameters(),      # 指定需要优化的模型参数
    lr=learning_rate         # 使用设定的学习率
)

### 5. 进入训练循环

In [32]:
num_epochs = 10  # 设置训练轮数
for epoch in range(num_epochs):  # 遍历每一轮训练
    for i, (images, labels) in enumerate(train_loader):  # 按批次读取训练图片和标签
        images = images.reshape(-1, 28 * 28)   # 将 28×28 图片展开为 784 维向量
        outputs = model(images)                # 前向传播，得到模型预测结果
        loss = criterion(outputs, labels)      # 计算预测结果与真实标签之间的损失
        optimizer.zero_grad()                  # 清空上一轮计算得到的梯度
        loss.backward()                        # 反向传播，计算各参数的梯度
        optimizer.step()                       # 根据梯度更新模型参数
        if (i + 1) % 100 == 0:                 # 每训练 100 个批次输出一次训练信息
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

Epoch [1/10], Step [100/600], Loss: 0.2461
Epoch [1/10], Step [200/600], Loss: 0.2904
Epoch [1/10], Step [300/600], Loss: 0.2236
Epoch [1/10], Step [400/600], Loss: 0.2302
Epoch [1/10], Step [500/600], Loss: 0.1227
Epoch [1/10], Step [600/600], Loss: 0.1013
Epoch [2/10], Step [100/600], Loss: 0.1214
Epoch [2/10], Step [200/600], Loss: 0.1113
Epoch [2/10], Step [300/600], Loss: 0.1189
Epoch [2/10], Step [400/600], Loss: 0.1443
Epoch [2/10], Step [500/600], Loss: 0.0598
Epoch [2/10], Step [600/600], Loss: 0.0977
Epoch [3/10], Step [100/600], Loss: 0.0537
Epoch [3/10], Step [200/600], Loss: 0.0290
Epoch [3/10], Step [300/600], Loss: 0.1252
Epoch [3/10], Step [400/600], Loss: 0.0598
Epoch [3/10], Step [500/600], Loss: 0.0449
Epoch [3/10], Step [600/600], Loss: 0.0216
Epoch [4/10], Step [100/600], Loss: 0.0114
Epoch [4/10], Step [200/600], Loss: 0.0179
Epoch [4/10], Step [300/600], Loss: 0.1103
Epoch [4/10], Step [400/600], Loss: 0.0351
Epoch [4/10], Step [500/600], Loss: 0.0125
Epoch [4/10

### 6. 验证/测试

In [33]:
with torch.no_grad():  # 测试阶段不计算梯度，减少内存占用并提高运行效率
    correct = 0        # 记录预测正确的样本数量
    total = 0          # 记录测试样本总数
    for images, labels in test_loader:                 # 按批次读取测试图片和真实标签
        images = images.reshape(-1, 28 * 28)           # 将 28×28 图片展开为 784 维向量
        outputs = model(images)                        # 前向传播，得到模型输出
        _, predicted = torch.max(outputs.data, 1)      # 取输出最大值对应的类别作为预测结果
        total += labels.size(0)                        # 累加当前批次的样本数量
        correct += (predicted == labels).sum().item()  # 累加预测正确的样本数量
    print(f'Accuracy of the network on the 10000 test images: {100 * correct / total} %')  # 输出测试准确率

Accuracy of the network on the 10000 test images: 98.16 %


### 7. 保存模型

In [34]:
torch.save(model, "../../mnist_mlp_model.pkl")  # 将训练好的整个 MLP 模型保存到本地文件